# 02 — Chunking, Embeddings and Indexing

**Propósito:** segmentar `medqa_train.parquet`, comparar `A_256`, `B_512` y `C_1024`, generar embeddings y construir los índices FAISS/BM25 para validación.

**Entrada requerida**
- `medqa_train.parquet` — generado por el Notebook 01.

**Salidas generadas**
- `df_chunks_A_256.parquet`
- `df_chunks_B_512.parquet`
- `df_chunks_C_1024.parquet`
- `embeddings_A_256.npy`
- `embeddings_B_512.npy`
- `embeddings_C_1024.npy`

> Los índices FAISS y BM25 se reconstruyen en memoria; los artefactos persistentes compartidos entre notebooks son los `.parquet` y `.npy`.


In [ ]:
!pip install -q pandas pyarrow matplotlib langchain-text-splitters sentence-transformers faiss-cpu rank_bm25

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

required = Path("medqa_train.parquet")
if not required.exists():
    raise FileNotFoundError(
        "Falta medqa_train.parquet. Ejecuta primero el Notebook 01 o carga el archivo "
        "en el directorio actual de esta sesión."
    )

train = pd.read_parquet(required)
print(f"medqa_train.parquet cargado: {len(train):,} filas")
print(f"Columnas: {list(train.columns)}")


# 1. Chunking + comparación de tamaños


El análisis previo reveló que 4,083 documentos (más del 25% del corpus de entrenamiento)
superan los 1,500 chars, límite aproximado del modelo `all-mpnet-base-v2`. Sin embargo,
aplicar chunking solo a ese subconjunto no es suficiente: documentos que están justo
por debajo del límite (ej. 1,400 chars) también pueden generar chunks poco precisos
para recuperación, ya que un chunk muy largo tiende a recuperar información irrelevante
junto con la relevante.

Por ello se decidió aplicar chunking a **todo el corpus**, comparando tres estrategias
con distintos tamaños y overlaps proporcionales:

- **Estrategia A** — `chunk_size=256`, `overlap=32`: chunks pequeños, alta precisión,
  mayor fragmentación.
- **Estrategia B** — `chunk_size=512`, `overlap=64`: balance entre contexto y precisión.
- **Estrategia C** — `chunk_size=1024`, `overlap=128`: chunks grandes, más contexto,
  menor fragmentación pero mayor riesgo de recuperar ruido.

In [ ]:
# Chunking + Comparación de 3 tamaños
from langchain_text_splitters import RecursiveCharacterTextSplitter

estrategias = {
    'A_256' : RecursiveCharacterTextSplitter(chunk_size=256,  chunk_overlap=32,  length_function=len),
    'B_512' : RecursiveCharacterTextSplitter(chunk_size=512,  chunk_overlap=64,  length_function=len),
    'C_1024': RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=128, length_function=len),
}

# Aplicar chunking a todo el corpus
# NOTA: se agregan document_url, question_focus y document_id respecto a la versión
# anterior, para soportar la técnica de citación / trazabilidad de la fuente en cada
# respuesta (sección 7.2). Estas columnas ya existen en `train` desde la limpieza
# inicial y no se pierden en ningún paso previo.
def apply_chunking_full(df, splitter, strategy_name):
    records = []
    for _, row in df.iterrows():
        chunks = splitter.split_text(row['answer'])
        for i, chunk in enumerate(chunks):
            records.append({
                'chunk_text'          : chunk,
                'question'            : row['question'],
                'document_source'     : row['document_source'],
                'document_url'        : row['document_url'],
                'question_focus'      : row['question_focus'],
                'document_id'         : row['document_id'],
                'question_type'       : row['question_type'],
                'answer_is_generic'   : row['answer_is_generic'],
                'answer_needs_chunking': row['answer_needs_chunking'],
                'chunk_id'            : i,
                'n_chunks'            : len(chunks),
                'strategy'            : strategy_name,
            })
    return pd.DataFrame(records)

df_chunks = {}
for name, splitter in estrategias.items():
    df_chunks[name] = apply_chunking_full(train, splitter, name)
    print(f"Estrategia {name}: {len(df_chunks[name]):,} chunks")

In [ ]:
# Tabla comparativa
print(f"\n{'='*65}")
print(f"{'Estrategia':<12} {'Chunks totales':>15} {'Chars promedio':>15} {'Chars mediana':>15}")
print(f"{'='*65}")
for name, df_c in df_chunks.items():
    lens = df_c['chunk_text'].str.len()
    print(f"{name:<12} {len(df_c):>15,} {lens.mean():>15.0f} {lens.median():>15.0f}")
print(f"{'='*65}")

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
colores = ['steelblue', 'darkorange', 'seagreen']
limites = [256, 512, 1024]

for ax, (name, df_c), color, lim in zip(axes, df_chunks.items(), colores, limites):
    df_c['chunk_text'].str.len().hist(bins=40, ax=ax, color=color)
    ax.axvline(lim, color='red', linestyle='--', label=f'límite {lim}')
    ax.set_title(f'Estrategia {name}')
    ax.set_xlabel('Chars por chunk')
    ax.set_ylabel('Frecuencia')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Ejemplo visual del overlap en un documento largo
ejemplo = train[train['answer_needs_chunking']]['answer'].iloc[0]
print(f"\nEjemplo sobre un documento de {len(ejemplo):,} chars:")
for name, splitter in estrategias.items():
    chunks = splitter.split_text(ejemplo)
    print(f"\nEstrategia {name} → {len(chunks)} chunks")
    for i, c in enumerate(chunks[:2]):
        print(f"  Chunk {i}: {c[:100]}...")
    if len(chunks) > 1:
        overlap_real = len(set(chunks[0].split()) & set(chunks[1].split()))
        print(f"  Palabras compartidas entre chunk 0 y 1: {overlap_real}")

| Aspecto | Estrategia A (256) | Estrategia B (512) | Estrategia C (1024) |
|---|---|---|---|
| **Chunks totales** | 71,805 | 38,127 | 21,623 |
| **Chars promedio** | 234 | 439 | 758 |
| **Chars mediana** | 252 | 507 | 928 |
| **Fragmentación** | Alta — 9 chunks para un doc de 1,886 chars | Media — 5 chunks | Baja — 2 chunks |
| **Palabras compartidas entre chunks** | 11 | 18 | 40 |
| **Precisión en recuperación** | Alta — chunks específicos a la query | Media | Baja — recupera ruido junto con lo relevante |
| **Contexto por chunk** | Bajo | Medio | Alto — mayor probabilidad de respuesta completa |
| **Riesgo en corpus médico** | Corta explicaciones encadenadas | Riesgo moderado | Recupera información irrelevante |
| **Overlap (% del chunk)** | 32 chars (12.5%) | 64 chars (12.5%) | 128 chars (12.5%) |
| **Favorece** | Precision@k | Balance Precision@k / Recall@k | Recall@k |
| **Estrategia final** | ❌ | ✅ Seleccionada | ❌ |

Los resultados confirman el tradeoff esperado entre tamaño y fragmentación.
La Estrategia A genera 71,805 chunks — más del triple que la Estrategia C —
lo que significa que un mismo documento médico queda dividido en muchos más
fragmentos. El ejemplo concreto lo ilustra bien: un documento de 1,886 chars
produce 9 chunks con A, 5 con B y solo 2 con C.

**Tradeoff tamaño vs fragmentación**

Chunks pequeños (A) fragmentan respuestas médicas que suelen ser párrafos
estructurados donde cada oración depende de la anterior. Por ejemplo, una
explicación sobre "Beta-ureidopropionase deficiency" que comienza en el
Chunk 0 continúa en el Chunk 1 con "which are building blocks of DNA..." —
sin contexto previo, ese chunk recuperado por sí solo no tiene sentido completo.
Chunks grandes (C) preservan esa coherencia pero al precio de recuperar
información irrelevante junto con la relevante.

**Impacto en recuperación**

La Estrategia A favorece Precision@k: cada chunk recuperado es más específico
a la query, pero puede no contener la respuesta completa. La Estrategia C
favorece Recall@k: mayor probabilidad de que el chunk contenga toda la
información necesaria, pero FAISS puede recuperar chunks con mucho ruido.
La Estrategia B representa el balance entre ambos extremos, con 38,127 chunks
de 439 chars promedio — suficiente contexto sin sacrificar precisión.

**Rol del overlap**

El número de palabras compartidas entre chunks consecutivos confirma que el
overlap funciona correctamente y escala de forma consistente: 11 palabras en A,
18 en B y 40 en C. Esto significa que en la Estrategia C dos chunks consecutivos
comparten 40 palabras — nivel de redundancia que podría hacer que FAISS recupere
chunks casi idénticos para una misma query, desperdiciando slots del top-k.

El overlap del 12.5% del chunk_size en las tres estrategias no es arbitrario:
está dentro del rango recomendado de 10-20% y es suficiente para cubrir
oraciones completas en texto médico sin generar redundancia excesiva.

Por estas razones se seleccionó la **Estrategia B (512 chars, overlap 64)**
como base para el pipeline RAG, aunque esta decisión será validada
empíricamente en la etapa de evaluación con Recall@k.

# 2. Generación de embeddings


**Modelo de embeddings: `all-mpnet-base-v2`**

Se utilizó el modelo pre-entrenado `all-mpnet-base-v2` de la librería
`sentence-transformers`. Este modelo transforma cada chunk de texto en un
vector de 768 dimensiones que captura su significado semántico.

**Justificación de la elección:**

- **Calidad semántica**: está fine-tuned sobre más de 1 billón de pares de
  oraciones usando un objetivo contrastivo, lo que lo hace especialmente
  bueno para recuperación semántica — exactamente lo que necesita RAG.

- **Dimensionalidad**: produce vectores de 768 dimensiones, más ricos que
  modelos más ligeros como `all-MiniLM-L6-v2` (384 dims), lo que permite
  capturar matices del lenguaje médico.

- **Límite de tokens**: 384 tokens — compatible con los chunks de las tres
  estrategias definidas (256, 512 y 1024 chars), ya que ninguno supera
  ese límite.

- **Dominio**: aunque no es un modelo especializado en medicina, fue entrenado
  con datos de QA entre sus fuentes, lo que lo hace adecuado para un corpus
  de preguntas y respuestas como MedQuAD.

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-mpnet-base-v2')
print(f"Modelo        : all-mpnet-base-v2")
print(f"Max tokens    : {embedding_model.max_seq_length}")
print(f"Dimensiones   : 768")

In [ ]:
# Generar embeddings para las 3 estrategias
embeddings = {}
for name, df_c in df_chunks.items():
    print(f"\nGenerando embeddings para estrategia {name} ({len(df_c):,} chunks)...")
    embeddings[name] = embedding_model.encode(
        df_c['chunk_text'].tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    print(f"Shape: {embeddings[name].shape}")

In [ ]:
import os

# Guardar embeddings y df_chunks
for name, emb in embeddings.items():
    np.save(f'embeddings_{name}.npy', emb)
    df_chunks[name].to_parquet(f'df_chunks_{name}.parquet', index=False)
    print(f"[{name}] embeddings: {emb.shape} | chunks: {len(df_chunks[name]):,}")

In [ ]:
# Cargar embeddings y df_chunks
embeddings = {}
df_chunks  = {}

for name in ['A_256', 'B_512', 'C_1024']:
    embeddings[name] = np.load(f'embeddings_{name}.npy')
    df_chunks[name]  = pd.read_parquet(f'df_chunks_{name}.parquet')
    print(f"[{name}] embeddings: {embeddings[name].shape} | chunks: {len(df_chunks[name]):,}")

**¿Qué son los embeddings?**

Un embedding es una representación numérica de un texto en un espacio vectorial
de alta dimensión (768 dimensiones en este caso). Textos semánticamente similares
producen vectores cercanos en ese espacio, lo que permite comparar significados
en lugar de palabras exactas.

**¿Por qué son necesarios en RAG?**

En RAG, la recuperación no puede basarse en búsqueda exacta de palabras clave
porque la query del usuario raramente usa las mismas palabras que el documento.
Por ejemplo, la query *"how is this condition passed down?"* debe recuperar
chunks que hablen de herencia genética aunque no contengan esas palabras exactas.
Los embeddings hacen posible esa búsqueda semántica.

# 3. Indexación con FAISS y BM25


**¿Qué es búsqueda vectorial?**

La búsqueda vectorial consiste en encontrar los vectores más cercanos a un
vector de consulta en un espacio de alta dimensión. En este sistema, cada
chunk de texto fue convertido en un vector de 768 dimensiones por el modelo
`all-mpnet-base-v2`. Cuando el usuario hace una pregunta, esa pregunta también
se convierte en un vector y FAISS busca los chunks cuyos vectores son más
similares — es decir, semánticamente más cercanos a la pregunta.

La métrica utilizada es **similitud coseno** (IndexFlatIP con normalización L2),
que mide el ángulo entre vectores en lugar de la distancia euclidiana. Esto es
consistente con la métrica con la que fue entrenado `all-mpnet-base-v2`, y es
más adecuada para texto semántico donde importa la dirección del vector,
no su magnitud.

---

**Se implementaron 4 estrategias de indexación y retrieval:**

- **Flat IP**: búsqueda exacta, sirve como baseline
- **HNSW**: búsqueda aproximada, más eficiente en producción — BONUS
- **Hybrid Search (BM25 + FAISS)**: combina búsqueda léxica y semántica — BONUS
- **Reranker Cross-Encoder**: reordena los candidatos de FAISS con mayor precisión — BONUS

In [ ]:
import faiss
from rank_bm25 import BM25Okapi

# Índice Flat IP (similitud coseno) — baseline
indexes_flat = {}
for name, emb in embeddings.items():
    dim = emb.shape[1]

    emb_norm = emb.astype('float32').copy()
    faiss.normalize_L2(emb_norm)

    index = faiss.IndexFlatIP(dim)
    index.add(emb_norm)
    indexes_flat[name] = index
    print(f"[Flat IP] {name}: {index.ntotal:,} vectores indexados")

# Índice HNSW (búsqueda aproximada) — BONUS
indexes_hnsw = {}
for name, emb in embeddings.items():
    dim = emb.shape[1]

    emb_norm = emb.astype('float32').copy()
    faiss.normalize_L2(emb_norm)

    M = 32
    index = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = 200
    index.add(emb_norm)
    indexes_hnsw[name] = index
    print(f"[HNSW IP] {name}: {index.ntotal:,} vectores indexados | M={M} | efConstruction=200")

# Índice BM25 — BONUS
bm25_indexes = {}
for name, df_c in df_chunks.items():
    tokenized = [text.lower().split() for text in df_c['chunk_text'].tolist()]
    bm25_indexes[name] = BM25Okapi(tokenized)
    print(f"[BM25] {name}: {len(tokenized):,} documentos indexados")

**HNSW — Hierarchical Navigable Small World (BONUS)**

HNSW es un algoritmo de búsqueda aproximada de vecinos más cercanos (ANN)
que construye un grafo jerárquico de múltiples capas sobre los vectores
del índice.

**¿Cómo funciona?**

- En la capa superior hay pocos nodos con conexiones largas (búsqueda gruesa)
- En las capas inferiores hay más nodos con conexiones cortas (búsqueda fina)
- Durante la búsqueda, el algoritmo entra por la capa superior y desciende
  progresivamente hacia los vecinos más cercanos, sin comparar contra
  todos los vectores

**Comparación Flat vs HNSW:**

| | Flat IP | HNSW IP |
|---|---|---|
| Tipo | Búsqueda exacta | Búsqueda aproximada |
| Complejidad | O(n) | O(log n) |
| Precisión | 100% | ~95-99% |
| Velocidad | Lenta en corpus grandes | Rápida |
| Memoria | Menor | Mayor |
| Ideal para | Validación y baseline | Producción |

**Parámetros utilizados:**

- `M=32`: número de conexiones por nodo en el grafo. Valores mayores
  aumentan precisión pero consumen más memoria. Se eligió 32 como balance
  dentro del rango estándar de 16-64.
- `efConstruction=200`: controla la precisión durante la construcción
  del índice. Valores mayores construyen un mejor grafo pero tardan más.
  200 es un valor que garantiza buena calidad sin un costo excesivo de construcción.

---

**Hybrid Search — BM25 + FAISS (BONUS)**

La búsqueda híbrida combina dos enfoques complementarios:

- **BM25**: búsqueda léxica basada en frecuencia de términos. Es precisa
  cuando la query usa exactamente las mismas palabras que el documento —
  especialmente útil para términos médicos específicos como nombres de
  enfermedades o genes que pueden no estar bien representados en los embeddings.

- **FAISS**: búsqueda semántica basada en embeddings. Entiende sinónimos
  y paráfrasis pero puede fallar con términos muy específicos.

La combinación se hace normalizando ambos scores a [0,1] y ponderándolos
con `alpha=0.5` (igual peso). El parámetro alpha es ajustable — valores
mayores priorizan FAISS, valores menores priorizan BM25.

---

**Reranker Cross-Encoder**

El reranker opera en dos etapas:

1. **FAISS recupera top-20 candidatos** rápidamente
2. **Cross-encoder reordena los 20** con una puntuación más precisa

La diferencia clave con los bi-encoders (FAISS) es que el cross-encoder
procesa la query y el chunk **juntos** en una sola pasada, capturando
interacciones entre ambos textos que un bi-encoder no detecta. El costo
es velocidad — por eso se aplica solo sobre los top-20 candidatos y no
sobre todo el índice.

El modelo utilizado es `cross-encoder/ms-marco-MiniLM-L-6-v2`, entrenado
específicamente para ranking de pasajes en tareas de búsqueda de información.

# 4. Validación de retrieval top-k


In [ ]:
from sentence_transformers import CrossEncoder

# Reranker Cross-Encoder — BONUS
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print(f"[Reranker] cross-encoder/ms-marco-MiniLM-L-6-v2 cargado")

# Flat / HNSW
# NOTA: se agregan document_url y question_focus al resultado para soportar
# la citación de fuente/artículo en la respuesta final (sección 7.2).
def retrieve(query, index, df_c, model, k=5):
    query_emb = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(query_emb)
    distances, indices = index.search(query_emb, k)
    results = df_c.iloc[indices[0]].copy()
    results['score'] = distances[0]
    return results[['chunk_text', 'question', 'document_source', 'document_url', 'question_focus', 'score']]

# Hybrid Search
def hybrid_retrieve(query, faiss_index, bm25_index, df_c, model, k=5, alpha=0.5):
    n = len(df_c)

    query_emb = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(query_emb)
    faiss_scores, faiss_indices = faiss_index.search(query_emb, n)
    faiss_score_arr = np.zeros(n)
    for idx, score in zip(faiss_indices[0], faiss_scores[0]):
        if idx < n:
            faiss_score_arr[idx] = score
    faiss_min, faiss_max = faiss_score_arr.min(), faiss_score_arr.max()
    if faiss_max > faiss_min:
        faiss_score_arr = (faiss_score_arr - faiss_min) / (faiss_max - faiss_min)

    tokenized_query = query.lower().split()
    bm25_scores = bm25_index.get_scores(tokenized_query)
    bm25_min, bm25_max = bm25_scores.min(), bm25_scores.max()
    if bm25_max > bm25_min:
        bm25_scores = (bm25_scores - bm25_min) / (bm25_max - bm25_min)

    combined = alpha * faiss_score_arr + (1 - alpha) * bm25_scores
    top_indices = np.argsort(combined)[::-1][:k]
    results = df_c.iloc[top_indices].copy()
    results['score_hybrid'] = combined[top_indices]
    results['score_faiss']  = faiss_score_arr[top_indices]
    results['score_bm25']   = bm25_scores[top_indices]
    return results[['chunk_text', 'question', 'document_source', 'document_url', 'question_focus', 'score_hybrid', 'score_faiss', 'score_bm25']]

# Reranker
def retrieve_with_reranker(query, faiss_index, df_c, model, k_retrieve=20, k_final=5):
    query_emb = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(query_emb)
    distances, indices = faiss_index.search(query_emb, k_retrieve)
    candidates = df_c.iloc[indices[0]].copy()
    candidates['faiss_score'] = distances[0]
    pairs = [[query, chunk] for chunk in candidates['chunk_text'].tolist()]
    rerank_scores = reranker.predict(pairs)
    candidates['rerank_score'] = rerank_scores
    candidates = candidates.sort_values('rerank_score', ascending=False).head(k_final)
    return candidates[['chunk_text', 'question', 'document_source', 'document_url', 'question_focus', 'faiss_score', 'rerank_score']]

In [ ]:
# Prueba de todas las estrategias
query = "What are the symptoms of Bell's palsy?"
k = 5

print(f"Query: {query}\n")
for name in ['A_256', 'B_512', 'C_1024']:
    print(f"{'='*60}")
    print(f"[{name}] Flat IP — top-{k}")
    print(retrieve(query, indexes_flat[name], df_chunks[name], embedding_model, k=k).to_string())

    print(f"\n[{name}] HNSW IP — top-{k}")
    print(retrieve(query, indexes_hnsw[name], df_chunks[name], embedding_model, k=k).to_string())

    print(f"\n[{name}] Hybrid Search — top-{k}")
    print(hybrid_retrieve(query, indexes_flat[name], bm25_indexes[name], df_chunks[name], embedding_model, k=k).to_string())

    print(f"\n[{name}] Reranker — top-{k} de 20")
    print(retrieve_with_reranker(query, indexes_flat[name], df_chunks[name], embedding_model).to_string())

**¿Qué significa top-k?**

top-k define cuántos chunks se recuperan del índice para pasarlos al LLM
como contexto. Por ejemplo, con k=5, FAISS devuelve los 5 chunks más similares
a la query, ordenados por score de mayor a menor.

Existe un tradeoff claro:

| k pequeño (ej. k=3) | k grande (ej. k=10) |
|---|---|
| Más preciso | Más contexto |
| Menos ruido para el LLM | Mayor probabilidad de incluir la respuesta |
| Puede omitir información relevante | Puede incluir chunks irrelevantes |

Se eligió **k=5** por tres razones concretas:

1. **Naturaleza del corpus**: MedQuAD es un dataset de QA donde cada pregunta
   tiene una respuesta específica y acotada. No es un corpus de documentos
   largos donde la respuesta puede estar dispersa en muchos fragmentos —
   con k=5 es suficiente para cubrir la respuesta y sus variantes.

2. **Límite de contexto del LLM**: pasar demasiados chunks al LLM aumenta
   el ruido en el prompt y puede degradar la calidad de la respuesta generada.
   Con chunks de 512 chars y k=5, el contexto total es de ~2,560 chars —
   manejable para cualquier LLM moderno sin saturar la ventana de contexto.

3. **Diversidad suficiente**: con k=5 se recuperan chunks de potencialmente
   distintas fuentes (GHR, GARD, CDC), lo que reduce el riesgo de que el LLM
   reciba una visión sesgada del tema consultado.

